# 🧠 AI Logic Engine: General Knowledge & Small Talk Ingestion

### ⚡ Speed & Scope Update (v3.8):
- **Focus:** Social Interactions, Small Talk, Etiquette, and General Knowledge.
- **High Speed:** Generation limits ကို ညှိထားပြီး တစ်ခါထုတ်လျှင် အချက်အလက် (၂၀) ခုအထိ တိုးမြှင့်တောင်းဆိုထားသည်။
- **Instruction:** Runtime ကို `T4 GPU` ထားရန် မမေ့ပါနှင့်။ Step 2 တွင် JSON key တင်ပေးရပါမည်။

In [ ]:
# @title 🛠️ Step 1: Install & Imports
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Setup Ready.")

In [ ]:
# @title 🔑 Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Firebase Service Account Key (JSON) ကို Upload တင်ပါ...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("❌ JSON file လိုအပ်ပါသည်။")
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Memory Synced: {DATABASE_ID}")

In [ ]:
# @title 🧬 Step 3: Logic Utilities
STATE_FILE = "ingestion_state_v3_8.json"

def clean_id(text):
    if not text: return ""
    text = str(text).lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

def normalize_verb(v):
    v = str(v).strip().lower()
    is_a_set = ['is_a', 'is a', 'is type of', 'belongs to', 'category of', 'member of']
    if any(p == v for p in is_a_set): return 'is_a'
    return v.replace(' ', '_')

def save_progress(count):
    with open(STATE_FILE, "w") as f: json.dump({"count": count}, f)

def load_progress():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f: return json.load(f).get("count", 0)
        except: pass
    return 0

print("✅ Logic Engine Ready.")

In [ ]:
# @title 🤖 Step 4: Load Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model.eval()
print("✅ AI Model Active (Optimized for Speed).")

In [ ]:
# @title 🚀 Step 5: Start Mass Ingestion
TARGET_GOAL = 100000 # @param {type:"number"}
CATEGORIES = [
    "Small Talk", "Social Etiquette", "Common Jokes", "Daily Manners", 
    "General Knowledge", "World Landmarks", "Pop Culture", "Cooking Basics",
    "Meeting Greetings", "Office Conversations", "Travel Phrases"
]

current_total = load_progress()
pbar = tqdm(initial=current_total, total=TARGET_GOAL, desc="Ingesting Symbols")

while current_total < TARGET_GOAL:
    try:
        cat = CATEGORIES[current_total % len(CATEGORIES)]
        
        # Faster Generation Prompt
        prompt = f"Extract 20 exhaustive facts about {cat}. Format: S|V|O|Sentence. No intro, no headers, PIPE ONLY."
        messages = [{"role": "system", "content": "Output S|V|O|Sentence ONLY."}, {"role": "user", "content": prompt}]
        
        it = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(it, return_tensors="pt", padding=True).to(model.device)
        
        with torch.no_grad():
            out = model.generate(inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=600, do_sample=True, temperature=0.7, pad_token_id=tokenizer.pad_token_id)
        
        resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        
        batch = db.batch()
        added = 0
        for line in resp.strip().split('\n'):
            if '|' not in line: continue
            pts = [x.strip() for x in line.split('|')]
            if len(pts) >= 3:
                s, v, o = pts[0], normalize_verb(pts[1]), pts[2]
                sid, oid = clean_id(s), clean_id(o)
                
                if sid and oid and sid != oid:
                    ref = db.collection('nodes').document(sid)
                    data = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    if v == 'is_a': data['groups'] = firestore.ArrayUnion([oid])
                    else:
                        sen = pts[3] if len(pts)>3 else f"{s} {v.replace('_',' ')} {o}."
                        data['relations'] = firestore.ArrayUnion([{'verb': v, 'targetId': oid, 'sentence': sen, 'weight': 100}])
                    batch.set(ref, data, merge=True)
                    added += 1
        
        if added > 0:
            batch.commit()
            current_total += added
            pbar.update(added)
            save_progress(current_total)
            
        if current_total % 30 == 0: torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"\n⚠️ Waiting: {e}")
        time.sleep(5)
    except KeyboardInterrupt: break

pbar.close()
print(f"✅ Sync Complete. Total Database Nodes: {current_total}")